# DR-GradCAM-Evaluation: Genuine Experimental Pipeline

This notebook executes the **genuine, non-destructive inference pipeline** for:
1. **IDRiD Domain Shift Diagnostic** ($N=413$ training images)
2. **Phase 6 Grad-CAM Lesion Localization** ($N=81$ segmentation images: 78 Primary + 3 Supplementary)
3. **Phase 7 Decision Faithfulness Evaluation** (Progressive deletion test with valid retinal FOV parity)

> **Backbone Checkpoint:** `experiments/checkpoints/best_model_qwk.pth` (Trained strictly on APTOS 2019, $N=2,929$)
> **Integrity Guarantees:** 100% frozen weights, zero optimizer updates, and dual immutability verification (SHA-256 + bit-exact `state_dict`).

### Cell 1: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# Clone repository
!git clone https://github.com/Shrees1ngh/DR-GradCAM-Evaluation.git
%cd DR-GradCAM-Evaluation
!git pull origin main

### Cell 2: Install Required Dependencies

In [ ]:
!pip install -r requirements.txt

### Cell 3: Link Trained Checkpoint & Validate Raw IDRiD Dataset (Assertion Gate 1)

In [ ]:
import os
import shutil
import hashlib
from pathlib import Path

os.makedirs('experiments/checkpoints', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)

# 1. Link Checkpoint
DRIVE_CKPT = '/content/drive/MyDrive/DR-GradCAM-Evaluation/best_model_qwk.pth'
LOCAL_CKPT = Path('experiments/checkpoints/best_model_qwk.pth')

if os.path.exists(DRIVE_CKPT):
    shutil.copy(DRIVE_CKPT, str(LOCAL_CKPT))
    print(f'✅ Checkpoint copied to {LOCAL_CKPT}')

assert LOCAL_CKPT.exists() and LOCAL_CKPT.stat().st_size > 0, (
    f'❌ Checkpoint missing or empty at {LOCAL_CKPT}! Please place best_model_qwk.pth in Google Drive.'
)
with open(LOCAL_CKPT, 'rb') as f:
    initial_sha = hashlib.sha256(f.read()).hexdigest()
print(f'✅ Checkpoint assertion passed: {LOCAL_CKPT} ({LOCAL_CKPT.stat().st_size:,} bytes)')
print(f'   Initial Checkpoint SHA-256: {initial_sha}')

# 2. Unzip or verify raw IDRiD dataset
DRIVE_IDRID_ZIP = '/content/drive/MyDrive/DR-GradCAM-Evaluation/IDRiD.zip'
if os.path.exists(DRIVE_IDRID_ZIP):
    !unzip -q {DRIVE_IDRID_ZIP} -d data/raw/
    print('✅ IDRiD raw dataset unzipped to data/raw/')

# Strict Input Assertion Gate: Check raw directories and counts
raw_grading_train = list(Path('data/raw/IDRiD/B. Disease Grading/B. Disease Grading/1. Original Images/a. Training Set').glob('*.jpg'))
raw_grading_test = list(Path('data/raw/IDRiD/B. Disease Grading/B. Disease Grading/1. Original Images/b. Testing Set').glob('*.jpg'))
raw_seg_train = list(Path('data/raw/IDRiD/A. Segmentation/A. Segmentation/1. Original Images/a. Training Set').glob('*.jpg'))
raw_seg_test = list(Path('data/raw/IDRiD/A. Segmentation/A. Segmentation/1. Original Images/b. Testing Set').glob('*.jpg'))

print(f'Raw IDRiD counts found:')
print(f' - Grading Training: {len(raw_grading_train)} (expected 413)')
print(f' - Grading Testing:  {len(raw_grading_test)} (expected 103)')
print(f' - Seg Training:     {len(raw_seg_train)} (expected 54)')
print(f' - Seg Testing:      {len(raw_seg_test)} (expected 27)')

assert len(raw_grading_train) == 413, f'Expected 413 grading train images, found {len(raw_grading_train)}'
assert len(raw_grading_test) == 103, f'Expected 103 grading test images, found {len(raw_grading_test)}'
assert len(raw_seg_train) + len(raw_seg_test) == 81, f'Expected 81 segmentation images, found {len(raw_seg_train) + len(raw_seg_test)}'
print('✅ All raw IDRiD input assertion gates passed successfully!')

### Cell 4: Preprocess IDRiD Images & FOV Quality Control (Assertion Gate 2)

In [ ]:
import sys
import subprocess
import pandas as pd
from pathlib import Path

# Pre-run cleanup of stale outputs to prevent false passes
stale_files = [
    Path('data/processed/idrid/external_train.csv'),
    Path('data/processed/idrid/external_test.csv'),
    Path('data/processed/idrid/segmentation_manifest.csv'),
    Path('reports/fov_qc_report.csv'),
    Path('reports/fov_qc_report.md'),
    Path('reports/fov_montage_81.png'),
    Path('reports/preprocessing_report.md'),
]
for p in stale_files:
    if p.exists():
        p.unlink()

# Run genuine preprocessing in --idrid-only mode with --force-reprocess and fail-fast check=True
cmd_prep = [
    sys.executable, 'scripts/preprocess_data.py',
    '--size', '512',
    '--idrid-only',
    '--validation-mode', 'strict',
    '--force-reprocess',
]
print(f"Executing: {' '.join(cmd_prep)}")
subprocess.run(cmd_prep, check=True)

# Post-Preprocessing Strict Validation Gate
df_train = pd.read_csv('data/processed/idrid/external_train.csv')
df_test = pd.read_csv('data/processed/idrid/external_test.csv')
df_seg = pd.read_csv('data/processed/idrid/segmentation_manifest.csv')

assert len(df_train) == 413, f'Expected 413 processed train rows, found {len(df_train)}'
assert len(df_test) == 103, f'Expected 103 processed test rows, found {len(df_test)}'
assert len(df_seg) == 81, f'Expected 81 processed seg rows, found {len(df_seg)}'

c_train = int((df_seg['cohort_origin'] == 'idrid_grading_train_cohort').sum())
c_test = int((df_seg['cohort_origin'] == 'idrid_grading_test_cohort').sum())
c_unique = int((df_seg['cohort_origin'] == 'idrid_seg_unique').sum())

assert c_train == 57, f'Expected 57 train-matched images, found {c_train}'
assert c_test == 3, f'Expected 3 test-matched images, found {c_test}'
assert c_unique == 21, f'Expected 21 unique images, found {c_unique}'

# Verify 81 FOV masks exist on disk and are non-empty
for _, r in df_seg.iterrows():
    fov_p = Path(r['mask_fov'])
    assert fov_p.exists() and fov_p.stat().st_size > 0, f'Missing FOV mask for {r["image_id"]}'

# Verify FOV Quality Control report and contact sheet montage exist
df_qc = pd.read_csv('reports/fov_qc_report.csv')
assert len(df_qc) == 81, f'Expected 81 QC evaluated records, found {len(df_qc)}'
assert (df_qc['status'] == 'PASSED').all(), f'Flagged anomalous FOV masks detected: {df_qc[df_qc["status"] != "PASSED"]}'

montage_p = Path('reports/fov_montage_81.png')
assert montage_p.exists() and montage_p.stat().st_size > 0, f'Missing FOV contact sheet montage at {montage_p}'
print('✅ Preprocessing & FOV Quality Control assertions passed: exactly 81 valid masks, 9x9 contact sheet generated.')

### Cell 5: Checkpoint Integrity & Architecture Verification (Assertion Gate 3)

In [ ]:
import hashlib
import yaml
import torch
from pathlib import Path
from src.model import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

# 1. Verify Checkpoint SHA-256
ckpt_path = Path('experiments/checkpoints/best_model_qwk.pth')
with open(ckpt_path, 'rb') as f:
    initial_sha = hashlib.sha256(f.read()).hexdigest()
print(f'Initial Checkpoint SHA-256: {initial_sha}')

# 2. Reconstruct Model architecture from configs/config.yaml without downloading ImageNet weights
config_path = Path('configs/config.yaml')
assert config_path.exists(), f'Missing config file at {config_path}'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

model = build_model(config=config, pretrained=False)

# 3. Load checkpoint state dict strictly from model_state_dict key
checkpoint_data = torch.load(ckpt_path, map_location=device)
if isinstance(checkpoint_data, dict):
    if 'model_state_dict' in checkpoint_data:
        state_dict = checkpoint_data['model_state_dict']
    elif 'state_dict' in checkpoint_data:
        state_dict = checkpoint_data['state_dict']
    else:
        state_dict = checkpoint_data
else:
    state_dict = checkpoint_data

model.load_state_dict(state_dict, strict=True)

# 4. Freeze all parameters strictly and verify architecture
model.eval()
for p in model.parameters():
    p.requires_grad = False
model.to(device)

num_tensors = len(model.state_dict())
print(f'✅ Reconstructed architecture and verified {num_tensors} parameter tensors match checkpoint strictly.')
print('   Checkpoint verified ready for evaluation subprocesses.')

### Cell 6: Run Genuine Domain Shift Diagnostic (N=413 IDRiD Train Images)

In [ ]:
import sys
import subprocess
import pandas as pd
from pathlib import Path

# Pre-run cleanup of diagnostic outputs to prevent false passes
diag_files = [
    Path('reports/idrid_training_predictions.csv'),
    Path('reports/idrid_training_domain_diagnostic_report.md'),
    Path('reports/idrid_training_domain_diagnostic_report.json'),
]
for p in diag_files:
    if p.exists():
        p.unlink()

# Run domain diagnostic via subprocess with check=True
cmd_diag = [
    sys.executable, 'scripts/diagnose_idrid_domain.py',
    '--checkpoint', 'experiments/checkpoints/best_model_qwk.pth',
    '--idrid-manifest', 'data/processed/idrid/external_train.csv',
]
print(f"Executing: {' '.join(cmd_diag)}")
subprocess.run(cmd_diag, check=True)

# Assert Diagnostic Outputs
df_diag = pd.read_csv('reports/idrid_training_predictions.csv')
assert len(df_diag) == 413, f'Expected 413 diagnostic predictions, got {len(df_diag)}'
assert Path('reports/idrid_training_domain_diagnostic_report.md').exists(), 'Missing diagnostic markdown report'
assert Path('reports/idrid_training_domain_diagnostic_report.json').exists(), 'Missing diagnostic JSON report'
print(f'✅ Domain Shift Diagnostic finished: {len(df_diag)} records evaluated.')

### Cell 7: Run Genuine Phase 6 Localization Evaluation (N=81 Images: 78 Primary + 3 Supplementary)

In [ ]:
import sys
import json
import subprocess
import pandas as pd
from pathlib import Path

# Pre-run cleanup of Phase 6 localization outputs to prevent false passes
loc_files = [
    Path('reports/phase_6_localization_predictions.csv'),
    Path('reports/phase_6_localization_report.md'),
    Path('reports/phase_6_localization_report.json'),
]
for p in loc_files:
    if p.exists():
        p.unlink()

# Run localization evaluation via subprocess with check=True
cmd_loc = [
    sys.executable, 'scripts/evaluate_localization.py',
    '--checkpoint', 'experiments/checkpoints/best_model_qwk.pth',
    '--manifest', 'data/processed/idrid/segmentation_manifest.csv',
    '--matches-json', 'data/idrid_seg_grading_matches.json',
]
print(f"Executing: {' '.join(cmd_loc)}")
subprocess.run(cmd_loc, check=True)

# Assert Localization Outputs
assert Path('reports/phase_6_localization_report.md').exists(), 'Missing phase 6 localization markdown report'
assert Path('reports/phase_6_localization_report.json').exists(), 'Missing phase 6 localization JSON report'

with open('reports/phase_6_localization_report.json', 'r', encoding='utf-8') as f:
    loc_data = json.load(f)

loc_results = loc_data.get('results', [])
assert len(loc_results) == 81, f'Expected 81 localization results, got {len(loc_results)}'

# Check cohort separation: 78 primary (57 matched + 21 unique) vs 3 supplementary (test matched)
primary_loc = [r for r in loc_results if r.get('cohort_origin') != 'idrid_grading_test_cohort']
supp_loc = [r for r in loc_results if r.get('cohort_origin') == 'idrid_grading_test_cohort']
assert len(primary_loc) == 78, f'Expected 78 primary localization records, got {len(primary_loc)}'
assert len(supp_loc) == 3, f'Expected 3 supplementary localization records, got {len(supp_loc)}'

# Verify per-image predictions CSV export
loc_csv_path = Path('reports/phase_6_localization_predictions.csv')
assert loc_csv_path.exists(), f'Missing localization predictions CSV at {loc_csv_path}'
df_loc = pd.read_csv(loc_csv_path)
assert len(df_loc) == 81, f'Expected 81 localization predictions in CSV, got {len(df_loc)}'
print('✅ Localization Evaluation assertion passed: 78 primary (57 matched + 21 unique), 3 supplementary.')

### Cell 8: Run Genuine Phase 7 Faithfulness Evaluation & Subprocess Immutability Verification

In [ ]:
import sys
import json
import subprocess
import hashlib
import pandas as pd
from pathlib import Path

# Pre-run cleanup of Phase 7 faithfulness outputs to prevent false passes
faith_files = [
    Path('reports/faithfulness_predictions.csv'),
    Path('reports/phase_7_faithfulness_report.md'),
    Path('reports/phase_7_faithfulness_report.json'),
]
for p in faith_files:
    if p.exists():
        p.unlink()

# Run faithfulness evaluation via subprocess with check=True
cmd_faith = [
    sys.executable, 'scripts/evaluate_faithfulness.py',
    '--checkpoint', 'experiments/checkpoints/best_model_qwk.pth',
    '--manifest', 'data/processed/idrid/segmentation_manifest.csv',
    '--matches-json', 'data/idrid_seg_grading_matches.json',
    '--deletion-steps', '0.10,0.20,0.30',
    '--fill-strategies', 'black,gaussian_blur',
    '--num-random-trials', '5',
]
print(f"Executing: {' '.join(cmd_faith)}")
subprocess.run(cmd_faith, check=True)

# Assert Faithfulness Outputs
assert Path('reports/phase_7_faithfulness_report.md').exists(), 'Missing phase 7 faithfulness markdown report'
assert Path('reports/phase_7_faithfulness_report.json').exists(), 'Missing phase 7 faithfulness JSON report'

df_faith = pd.read_csv('reports/faithfulness_predictions.csv')
assert len(df_faith) == 81, f'Expected 81 faithfulness predictions, got {len(df_faith)}'

# Subprocess Immutability Verification:
# The evaluation scripts verify their own model weights during execution in their separate processes.
with open('reports/phase_6_localization_report.json', 'r', encoding='utf-8') as f:
    loc_json = json.load(f)
assert loc_json.get('state_dict_verified') is True, 'Phase 6 subprocess failed internal state_dict verification!'
assert loc_json.get('checkpoint_sha256') == initial_sha, 'Phase 6 checkpoint SHA-256 mismatch!'

with open('reports/phase_7_faithfulness_report.json', 'r', encoding='utf-8') as f:
    faith_json = json.load(f)
assert faith_json.get('state_dict_verified') is True, 'Phase 7 subprocess failed internal state_dict verification!'
assert faith_json.get('checkpoint_sha256') == initial_sha, 'Phase 7 checkpoint SHA-256 mismatch!'

# Verify checkpoint file SHA-256 on disk before and after all subprocess executions
with open(ckpt_path, 'rb') as f:
    final_sha = hashlib.sha256(f.read()).hexdigest()
assert initial_sha == final_sha, f'Checkpoint file on disk was modified! Initial: {initial_sha}, Final: {final_sha}'

print('✅ Subprocess Immutability Verified: Both Phase 6 and Phase 7 verified bit-exact weights in their runtime processes.')
print(f'✅ Checkpoint File SHA-256 Unaltered: {final_sha}')

### Cell 9: Sync Reports and Predictions Back to Google Drive

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_REPORTS = '/content/drive/MyDrive/DR-GradCAM-Evaluation/reports'
os.makedirs(DRIVE_REPORTS, exist_ok=True)

# Verify core report files exist prior to backup
required_reports = [
    'reports/idrid_training_domain_diagnostic_report.md',
    'reports/idrid_training_domain_diagnostic_report.json',
    'reports/idrid_training_predictions.csv',
    'reports/phase_6_localization_report.md',
    'reports/phase_6_localization_report.json',
    'reports/phase_6_localization_predictions.csv',
    'reports/phase_7_faithfulness_report.md',
    'reports/phase_7_faithfulness_report.json',
    'reports/faithfulness_predictions.csv',
    'reports/fov_qc_report.csv',
    'reports/fov_qc_report.md',
    'reports/fov_montage_81.png',
]
for rp in required_reports:
    p = Path(rp)
    assert p.exists() and p.stat().st_size > 0, f'Report {rp} missing or empty!'

!cp -r reports/* {DRIVE_REPORTS}/
print(f'✅ All {len(required_reports)} reports and predictions successfully verified and backed up to {DRIVE_REPORTS}!')

### Cell 10: Summary of Genuine Empirical Findings

In [ ]:
import json
import pandas as pd

df_diag = pd.read_csv('reports/idrid_training_predictions.csv')
with open('reports/phase_6_localization_report.json', 'r', encoding='utf-8') as f:
    loc_json = json.load(f)
df_faith = pd.read_csv('reports/faithfulness_predictions.csv')

print('=' * 65)
print('GENUINE EXPERIMENTAL PIPELINE SUMMARY')
print('=' * 65)
acc = (df_diag['true_grade'] == df_diag['predicted_grade']).mean() * 100
print(f'1. IDRiD Diagnostic (N={len(df_diag)}): Accuracy = {acc:.2f}%')

loc_summary = loc_json.get('summary', {})
pri_summary = loc_summary.get('primary_cohort', {})
pri_cnt = pri_summary.get('total_images', 0)
supp_summary = loc_summary.get('supplementary_test_cohort', {})
supp_cnt = supp_summary.get('total_images', 0)
pri_iou = pri_summary.get('composite_iou', {}).get('mean', 0.0)
pri_dice = pri_summary.get('composite_dice', {}).get('mean', 0.0)
print(f'2. Phase 6 Localization (N=81): Primary={pri_cnt} (Mean IoU={pri_iou:.4f}, Dice={pri_dice:.4f}) | Supp={supp_cnt}')
print(f'3. Phase 7 Faithfulness (N={len(df_faith)}): Total evaluated records = {len(df_faith)}')
print('=' * 65)
display(df_faith.head(10))
